# 05 — RAG corpus embedding (Plan 3.0)

**Authored by:** Role C — AI / ML / Voice Lead

Reads [`ml/rag/corpus.jsonl`](../rag/corpus.jsonl) (30 sourced snippets — WHO IMCI, India MoHFW STG, NICE CKS, WHO mhGAP) and writes `ml/rag/embeddings.jsonl` with BGE-M3 1024-dim multilingual vectors. Role B picks up `embeddings.jsonl` and upserts to Supabase `pgvector` via `backend/app/nlp/rag.py`.

Shared implementation lives in [`ml/rag/embed.py`](../rag/embed.py) — this notebook is the runnable mirror used during development. Run end-to-end with: `py -3.12 ml/rag/embed.py`.

In [ ]:
import sys, pathlib, json
REPO = pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / 'ml' / 'rag'))
from embed import load_corpus, embed_with_bge, write_embeddings, DEFAULT_CORPUS, DEFAULT_OUT, DEFAULT_MODEL
rows = load_corpus(DEFAULT_CORPUS)
print(f'loaded {len(rows)} snippets')
print('tags distribution:')
from collections import Counter
tags = Counter(t for r in rows for t in r.get('tags', []))
for tag, n in tags.most_common(15):
    print(f'  {tag:20s} {n}')

In [ ]:
# Embed all 30 snippets with BGE-M3 (1024-dim, multilingual)
enriched = embed_with_bge(rows, DEFAULT_MODEL)
print(f'embedded {len(enriched)} snippets, dim={enriched[0]["embedding_dim"]}')
write_embeddings(enriched, DEFAULT_OUT)
print(f'wrote {DEFAULT_OUT}')

In [ ]:
# Sanity-check retrieval — top-3 cosine nearest neighbours for a few real triage queries
import numpy as np
from sentence_transformers import SentenceTransformer
M = SentenceTransformer(DEFAULT_MODEL)
E = np.array([r['embedding'] for r in enriched])  # already L2-normalised

def topk(query, k=3):
    q = M.encode([query], normalize_embeddings=True)[0]
    scores = E @ q
    idx = np.argsort(-scores)[:k]
    return [(enriched[i]['id'], enriched[i]['source'], float(scores[i])) for i in idx]

for q in [
    'severe chest pain radiating to left arm in a 67yo diabetic',
    'one side of my face is drooping and my arm feels weak',
    'my child is 3 years old and has fever 39.5 and is lethargic',
    'asthma attack cannot finish sentence SpO2 87',
    'vaginal bleeding 8 weeks pregnant dizzy',
    'I dont want to live anymore',
]:
    print(f'\nQ: {q}')
    for cid, src, score in topk(q, k=3):
        print(f'  {score:.3f}  {cid}  ({src})')

### Handoff to Role B

Once `ml/rag/embeddings.jsonl` exists, Role B runs the pgvector upsert from [`backend/app/nlp/rag.py`](../../backend/app/nlp/rag.py):

```bash
py -3.12 -c "from backend.app.nlp.rag import upsert_corpus; upsert_corpus()"
```

and verifies retrieval against the same sanity queries above via `retrieve('chest pain', k=3)`.